In [ ]:
import pandas as pd

CAMINHO_PADRAO = "dados/vgsales.csv"
CAMINHO_SAIDA = "dados/vgsales_tratado.csv"

RENOMEAR_COLUNAS = {
    "Rank": "rank_mundial",
    "Name": "nome",
    "Platform": "plataforma",
    "Year": "ano",
    "Genre": "genero",
    "Publisher": "publisher",
    "NA_Sales": "vendas_na",
    "EU_Sales": "vendas_eu",
    "JP_Sales": "vendas_jp",
    "Other_Sales": "vendas_outras",
    "Global_Sales": "vendas_global",
}

COLUNAS_REGIOES = ["vendas_na", "vendas_eu", "vendas_jp", "vendas_outras"]

NOMES_REGIOES = {
    "vendas_na": "América do Norte",
    "vendas_eu": "Europa",
    "vendas_jp": "Japão",
    "vendas_outras": "Outras regiões",
}

GENERO_PT = {
    "Action": "Ação", "Sports": "Esportes", "Platform": "Plataforma", "Racing": "Corrida",
    "Role-Playing": "RPG", "Puzzle": "Quebra-cabeça", "Misc": "Diversos", "Shooter": "Tiro",
    "Simulation": "Simulação", "Fighting": "Luta", "Adventure": "Aventura", "Strategy": "Estratégia",
}

FABRICANTE_POR_PLATAFORMA = {
    **dict.fromkeys(["Wii", "NES", "GB", "DS", "SNES", "GBA", "3DS", "N64", "GC", "WiiU"], "Nintendo"),
    **dict.fromkeys(["PS", "PS2", "PS3", "PS4", "PSP", "PSV"], "Sony"),
    **dict.fromkeys(["X360", "XB", "XOne"], "Microsoft"),
    **dict.fromkeys(["GEN", "DC", "SAT", "SCD", "GG"], "Sega"),
    "2600": "Atari",
    "PC": "PC",
}

ANO_MAXIMO_VALIDO = 2016


def carregar_dados_brutos(caminho: str = CAMINHO_PADRAO) -> pd.DataFrame:

    return pd.read_csv(caminho)


def tratar_dados(bruto: pd.DataFrame) -> pd.DataFrame:

    df = bruto.rename(columns=RENOMEAR_COLUNAS).copy()

    for coluna in ["nome", "plataforma", "genero", "publisher"]:
        df[coluna] = df[coluna].where(df[coluna].isna(), df[coluna].astype(str).str.strip())

    df = df.drop_duplicates()

    df = df.dropna(subset=["ano"])
    df = df[df["ano"] <= ANO_MAXIMO_VALIDO]
    df["ano"] = df["ano"].astype(int)

    df["publisher"] = df["publisher"].fillna("Desconhecido")

    df["genero"] = df["genero"].map(GENERO_PT).fillna(df["genero"])
    df["fabricante"] = df["plataforma"].map(FABRICANTE_POR_PLATAFORMA).fillna("Outros")
    df["decada"] = (df["ano"] // 10 * 10).astype(str) + "s"
    df["nome_plataforma"] = df["nome"] + " (" + df["plataforma"] + ")"

    return df.reset_index(drop=True)


def formato_longo_regioes(df: pd.DataFrame) -> pd.DataFrame:


    longo = df.melt(
        id_vars=["nome", "plataforma", "ano", "genero", "fabricante"],
        value_vars=COLUNAS_REGIOES, var_name="regiao", value_name="vendas",
    )
    longo["regiao"] = longo["regiao"].map(NOMES_REGIOES)
    return longo


if __name__ == "__main__":
    bruto = carregar_dados_brutos()
    tratado = tratar_dados(bruto)

    print("=== RELATÓRIO DE TRATAMENTO ===")
    print(f"Dados brutos:   {bruto.shape[0]} linhas x {bruto.shape[1]} colunas")
    print(f"Dados tratados: {tratado.shape[0]} linhas x {tratado.shape[1]} colunas")
    print(f"Sem ano (removidos):     {bruto['Year'].isna().sum()}")
    print(f"Ano > {ANO_MAXIMO_VALIDO} (removidos):    {(bruto['Year'] > ANO_MAXIMO_VALIDO).sum()}")
    print(f"Publisher ausente (mantidos como 'Desconhecido'): {bruto['Publisher'].isna().sum()}")
    print("\nResumo das vendas (milhões de unidades):")
    print(tratado[["vendas_na", "vendas_eu", "vendas_jp", "vendas_outras", "vendas_global"]].describe().round(2).to_string())
    tratado.to_csv(CAMINHO_SAIDA, index=False)
    print(f"\nArquivo salvo em {CAMINHO_SAIDA}")